In [ ]:
# Import packages
import sys
import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

src_path = str(Path.cwd().parent)
if src_path not in sys.path:
    sys.path.append(src_path)
    
import microscopy_analysis.d00_utils.utilities as utils
import microscopy_analysis.d00_utils.dirnames as dn

%matplotlib notebook
%matplotlib inline

In [ ]:
df_path = Path(input('Please enter the full path for the dataframe:\n'))

In [ ]:
df = pd.read_csv(df_path)
df.head()
print(f'Num cells: {len(df)}')

In [ ]:
graphs_dir = None
if graphs_dir is None:
    graphs_dir = df_path.parent / 'graphs'
    graphs_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
conds = df['condition'].unique().tolist()
print(conds)

In [ ]:
df.columns.to_list()

In [ ]:
utils.safe_save_csv(df, df_path)

In [ ]:
df = df[df['omit']!=True]
print(f'Num cells: {len(df)}')

#df = df[df['experiment'] != 'CE019']
print(f'Num cells: {len(df)}')

In [ ]:
#Count number of datapoints in each category
counts = df.groupby(['condition', 'experiment', 'condition'])['UID'].count()
counts

In [ ]:
counts.to_csv(df_path.parent / 'imagecounts.csv')

In [ ]:
# Plot cell area graph

y_label = 'cell area'
x_label = 'condition'
graphname = 'cell_area.png'
hue = 'experiment'

x_order = ['Control', 'DeAct']
#sns.swarmplot(df, x=x_label, y='cell area', order=x_order, legend=False, color='gray')
sns.swarmplot(df, x=x_label, y=y_label, order=x_order, hue=hue, legend=True, size=3)
#plt.savefig(graphs_dir / graphname)
plt.show()

In [ ]:
# Plot compaction graph
y_label = '% compaction'
x_label = 'condition'
graphname = 'compaction.png'

hue = 'experiment'
x_order = ['Control', 'DeAct', 'StablAct', 'Ezrin']

sns.swarmplot(df, x=x_label, y=y_label, order=x_order, hue=hue, size=3)
#plt.savefig(graphs_dir / graphname)
plt.show()

In [ ]:
# Calculate averages
groupbycols = ['condition', 'experiment']
agg_cols = {'% compaction':'mean', 'cell area':'mean'}

ReplicateAverages = df.groupby(groupbycols, as_index=False).agg(agg_cols)
print("Replicate Averages:")
print(ReplicateAverages)

In [ ]:
avgs = df.groupby(['tx', 'experiment'], as_index=False).agg({'% compaction':'mean', 'cell area':'mean'})
avgs

In [ ]:
y_label = '% compaction'

ReplicateAvePivot = ReplicateAverages.pivot_table(columns = 'condition', values = y_label, index = "experiment")
print("Replicate Average Pivot:")
print(ReplicateAvePivot)

In [ ]:
var1 = 'Control'
var2 = 'DeAct'
statistic, pvalue = stats.ttest_rel(ReplicateAvePivot['Control'], ReplicateAvePivot['DeAct'])
P_value = str(float(round(pvalue, 3)))

print('\n')

print(f'P value comparing {y_label} between {var1} and {var2} groups: {P_value}')

In [ ]:


x_order = ['Control', 'DeAct']
sns.swarmplot(x = "condition", y = "% compaction", order=x_order, data = df, color='gray', size=2)
ax = sns.pointplot(x = "condition", y = "% compaction", order=x_order, hue = 'experiment', 
                   join = False, data = ReplicateAverages)
#ax.legend_.remove()

plt.savefig(graphs_dir / graphname)

In [ ]:
graphname = 'compaction_superplot.png'

x_order = ['Control', 'DeAct']
sns.swarmplot(x = "condition", y = "% compaction", order=x_order, data = df, color='gray', size=3)
sns.swarmplot(x = "condition", y = "% compaction", order=x_order, data = ReplicateAverages, color='k', size=6)
plt.ylim(0, 100)
#ax.legend_.remove()

plt.savefig(graphs_dir / graphname)

In [ ]:
deact_df = df[df['condition']=='DeAct']

In [ ]:
deact_df.columns

In [ ]:
deact_df